In [19]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

True

# Teste - Leitura dos links

In [3]:
html_doe = r'https://www.diariooficial.ms.gov.br/'

In [4]:
from bs4 import BeautifulSoup
import requests

response = requests.get(html_doe)
soup = BeautifulSoup(response.text, 'html.parser')

# Encontra todos os links que terminam em .pdf
links = soup.find_all('a', href=True)

for link in links:
    url = link['href']
    texto = link.get_text().lower()
    
    if ".pdf" in url:
        # Identifica o tipo sem baixar
        tipo = "Normal"
        if "extra" in texto or "extra" in url:
            tipo = "Extraordinário"
        elif "suplementar" in texto:
            tipo = "Suplementar"
            
        print(f"Encontrado: {tipo} | Link: {url}")
        
        # Só aqui você decide se chama a função de download/leitura
        # if condicao_de_data: 
        #    baixar_e_ler(url)

Encontrado: Extraordinário | Link: https://assets.imprensaoficial.ms.gov.br/public/prd/Diario%20Oficial/2026/04/17/DO12133_17_04_2026.pdf
Encontrado: Normal | Link: https://assets.imprensaoficial.ms.gov.br/public/prd/Diario%20Oficial/2026/04/17/DO12132_17_04_2026.pdf
Encontrado: Extraordinário | Link: https://assets.imprensaoficial.ms.gov.br/public/prd/Diario%20Oficial/2026/04/16/DO12131_16_04_2026.pdf
Encontrado: Normal | Link: https://assets.imprensaoficial.ms.gov.br/public/prd/Diario%20Oficial/2026/04/16/DO12130_16_04_2026.pdf
Encontrado: Extraordinário | Link: https://assets.imprensaoficial.ms.gov.br/public/prd/Diario%20Oficial/2026/04/15/DO12129_15_04_2026.pdf
Encontrado: Normal | Link: https://assets.imprensaoficial.ms.gov.br/public/prd/Diario%20Oficial/2026/04/15/DO12128_15_04_2026.pdf
Encontrado: Normal | Link: https://assets.imprensaoficial.ms.gov.br/public/prd/Diario%20Oficial/2026/04/15/DO12128_15_04_2026_SUP_1.pdf
Encontrado: Normal | Link: https://assets.imprensaoficial.ms

# Automação - Leitura e escrita dos PDFs

In [5]:
# Configurações
URL_SITE = r'https://www.diariooficial.ms.gov.br/' # Exemplo do DOE-MS
HISTORICO_CSV = os.path.join(os.getenv("PATH_HISTORICO_URL"), "historico.csv")
BASE_URL = "https://assets.imprensaoficial.ms.gov.br"

In [6]:
import pandas as pd 
pd.DataFrame(columns = ['url', 'pdf_name']).to_csv(HISTORICO_CSV, index =False)

In [7]:
import requests
from bs4 import BeautifulSoup
import os
import csv
import re

def carregar_historico_urls():
    """Lê o CSV e retorna um conjunto apenas com as URLs para comparação rápida."""
    if not os.path.exists(HISTORICO_CSV):
        return set()
    urls = set()
    with open(HISTORICO_CSV, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            urls.add(row['url'])
    return urls

def limpar_nome_tag(texto):
    """Extrai o que vem após o hífen e limpa para usar como nome de arquivo."""
    if "-" in texto:
        # Pega a última parte após o hífen
        tag = texto.split("-")[-1].strip()
        # Remove caracteres especiais e troca espaços por underline
        tag = re.sub(r'[^\w\s-]', '', tag).replace(" ", "_").lower()
        return f"_{tag}"
    return ""

def salvar_no_historico(novos_dados):
    """Adiciona novas linhas ao CSV."""
    arquivo_novo = not os.path.exists(HISTORICO_CSV)
    
    with open(HISTORICO_CSV, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['url', 'pdf_name'])
        if arquivo_novo:
            writer.writeheader()
        writer.writerows(novos_dados)

def buscar_links_atuais():
    """Acessa o site e retorna lista de dicionários com URL e Nome Sugerido."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(URL_SITE, headers=headers, timeout=20)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        resultados = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if '.pdf' in href.lower():
                texto_link = a.get_text(strip=True)
                
                # 1. URL Absoluta
                url_completa = href if href.startswith('http') else f"{BASE_URL}{href}"
                
                # 2. Lógica de nome flexível
                nome_base = url_completa.split('/')[-1].replace('.pdf', '')
                tag = limpar_nome_tag(texto_link)
                nome_final = f"{nome_base}{tag}.pdf"
                
                resultados.append({
                    'url': url_completa, 
                    'pdf_name': nome_final
                })
        return resultados
    except Exception as e:
        print(f"Erro ao acessar o site: {e}")
        return []

def executar_mapeamento():
    print("Iniciando mapeamento para CSV...")
    
    urls_vistas = carregar_historico_urls()
    dados_atuais = buscar_links_atuais()
    
    novos_registros = []
    for item in dados_atuais:
        if item['url'] not in urls_vistas:
            novos_registros.append(item)
            # Evita duplicados na mesma execução caso o site repita o link
            urls_vistas.add(item['url'])

    if novos_registros:
        print(f"Encontrados {len(novos_registros)} novos arquivos.")
        salvar_no_historico(novos_registros)
        for r in novos_registros:
            print(f" [+] Mapeado: {r['pdf_name']}")
    else:
        print("Nenhuma novidade no site.")

if __name__ == "__main__":
    executar_mapeamento()

Iniciando mapeamento para CSV...
Encontrados 10 novos arquivos.
 [+] Mapeado: DO12133_17_04_2026_edição_extra.pdf
 [+] Mapeado: DO12132_17_04_2026.pdf
 [+] Mapeado: DO12131_16_04_2026_edição_extra.pdf
 [+] Mapeado: DO12130_16_04_2026.pdf
 [+] Mapeado: DO12129_15_04_2026_edição_extra.pdf
 [+] Mapeado: DO12128_15_04_2026.pdf
 [+] Mapeado: DO12128_15_04_2026_SUP_1_detran.pdf
 [+] Mapeado: DO12127_14_04_2026.pdf
 [+] Mapeado: DO12127_14_04_2026_SUP_1_ses.pdf
 [+] Mapeado: DO12127_14_04_2026_SUP_2_detran.pdf


## Escrita dos PDFs

In [8]:
os.getenv("PATH_PDF")

'/home/gcioccia/Projects/Apps/monitor_doe/data/pdfs'

In [9]:
# Configurações
PASTA_DESTINO = os.path.join(os.getenv("PATH_PDF"))
HISTORICO_CSV = os.path.join(os.getenv("PATH_HISTORICO_URL"), "historico.csv")

In [10]:
import requests
import os
import csv

def baixar_arquivos():
    # 1. Cria a pasta se não existir
    if not os.path.exists(PASTA_DESTINO):
        os.makedirs(PASTA_DESTINO)
        print(f"Pasta '{PASTA_DESTINO}' criada.")

    # 2. Verifica se o CSV existe
    if not os.path.exists(HISTORICO_CSV):
        print(f"Erro: Arquivo '{HISTORICO_CSV}' não encontrado. Rode o mapeador primeiro.")
        return

    # 3. Lê os dados do CSV
    try:
        with open(HISTORICO_CSV, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            # Criamos uma lista de dicionários para iterar
            diarios = [row for row in reader]
    except Exception as e:
        print(f"Erro ao ler o CSV: {e}")
        return

    print(f"Iniciando verificação/download de {len(diarios)} arquivos...")

    for diario in diarios:
        url = diario['url']
        nome_arquivo = diario['pdf_name']
        caminho_completo = os.path.join(PASTA_DESTINO, nome_arquivo)

        # 4. Verificação de existência (usando o nome customizado com tag)
        if os.path.exists(caminho_completo):
            print(f"[-] Já existe localmente: {nome_arquivo}")
            continue

        # 5. Download com Stream
        try:
            print(f"[+] Baixando: {nome_arquivo}...")
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, timeout=60, stream=True)
            response.raise_for_status()

            with open(caminho_completo, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk: # filtra chunks vazios
                        f.write(chunk)
            
            print(f"    [OK] Salvo com sucesso.")
            
        except Exception as e:
            print(f"    [X] Erro ao baixar {nome_arquivo}: {e}")

if __name__ == "__main__":
    baixar_arquivos()

Iniciando verificação/download de 10 arquivos...
[-] Já existe localmente: DO12133_17_04_2026_edição_extra.pdf
[-] Já existe localmente: DO12132_17_04_2026.pdf
[-] Já existe localmente: DO12131_16_04_2026_edição_extra.pdf
[-] Já existe localmente: DO12130_16_04_2026.pdf
[-] Já existe localmente: DO12129_15_04_2026_edição_extra.pdf
[-] Já existe localmente: DO12128_15_04_2026.pdf
[-] Já existe localmente: DO12128_15_04_2026_SUP_1_detran.pdf
[-] Já existe localmente: DO12127_14_04_2026.pdf
[+] Baixando: DO12127_14_04_2026_SUP_1_ses.pdf...
    [OK] Salvo com sucesso.
[+] Baixando: DO12127_14_04_2026_SUP_2_detran.pdf...
    [OK] Salvo com sucesso.


# Enviar notificação

In [20]:
# Configurações de E-mail
EMAIL_REMETENTE = os.getenv("EMAIL_REMETENTE")
SENHA_APP = os.getenv("SENHA_EMAIL")
EMAIL_DESTINATARIO = "guilherme.cioccia@gmail.com"


In [22]:
SENHA_APP

'fnxl gaun cygt terc'

In [28]:
import smtplib
import os
from email.message import EmailMessage

def enviar_email_com_anexo(email_destinatario, caminho_completo_pdf, nome_arquivo):
    msg = EmailMessage()
    msg['Subject'] = f"[TESTE] Novo Diário Disponível: {nome_arquivo}"
    msg['From'] = EMAIL_REMETENTE
    msg['To'] = email_destinatario
    
    msg.set_content(f"""
    Olá,
    
    O sistema de monitoramento detectou uma nova publicação.
    
    Arquivo: {nome_arquivo}
    Status: Baixado e disponível na pasta local.
    """)

    try:
        # Lê o conteúdo do PDF
        with open(caminho_completo_pdf, 'rb') as f:
            file_data = f.read()
            
        # Adiciona o anexo
        msg.add_attachment(
            file_data,
            maintype='application',
            subtype='pdf',
            filename=nome_arquivo
        )

        # Conexão segura com o servidor do Gmail
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(EMAIL_REMETENTE, SENHA_APP)
            smtp.send_message(msg)
            
        print(f"✅ Notificação enviada: {nome_arquivo}")

    except Exception as e:
        print(f"❌ Falha ao enviar e-mail: {e}")


In [ ]:
email_destinatario = "guilherme.cioccia@gmail.com"

# --- TESTE RÁPIDO ---
if __name__ == "__main__":
    # Teste com um arquivo que você sabe que já baixou
    arquivo_teste = "/home/gcioccia/Projects/Apps/monitor_doe/data/pdfs/DO12133_17_04_2026_edição_extra.pdf"
    if os.path.exists(arquivo_teste):
        enviar_email_com_anexo(email_destinatario, arquivo_teste, os.path.basename(arquivo_teste))
    else:
        print("Aponte para um arquivo PDF existente na pasta downloads_doe para testar.")

✅ Notificação enviada: DO12133_17_04_2026_edição_extra.pdf
